In [ ]:
# ==============================
# IMPORT DES LIBRAIRIES
# ==============================
import pandas as pd


In [ ]:
# ==============================
# CHARGEMENT DU DATASET BRUT
# ==============================
# On charge le fichier CSV depuis le dossier raw
df = pd.read_csv("../data/raw/annecy_matches_2025_2026.tsv", sep="\t")

# ==============================
# INFORMATIONS SUR LE DATASET
# ==============================

# Permet de voir :
# - le nombre de lignes
# - les types de données
# - les valeurs manquantes
df.info()
print("\n--------------------------------")
print("Nombre de lignes et de colonnes :", df.shape)



In [ ]:
# ==============================
# NETTOYAGE DES NOMS DE COLONNES
# ==============================
# Suppression des espaces inutiles avant/après les noms de colonnes
# Cela évite les erreurs lors de l'accès aux colonnes (ex : " Date" vs "Date")

df.columns = df.columns.str.strip()

# Vérification des noms de colonnes
print(df.columns)

In [ ]:
# ==============================
# SUPPRESSION DES COLONNES INUTILES
# ==============================
# Certaines colonnes ne sont pas utiles pour l'analyse
# errors="ignore" évite une erreur si la colonne n'existe pas déjà
df = df.drop(columns=[
    "Match Report",
    "Notes",
], errors="ignore")

# ==============================
# CONVERSION DE LA DATE
# ==============================
# La colonne Date est actuellement du texte
# On la convertit en vrai format datetime
df["Date"] = pd.to_datetime(df["Date"])


# ==============================
# NETTOYAGE DE LA COLONNE ATTENDANCE
# ==============================
# Attendance = nombre de spectateurs présents au stade
# Exemple : "2,901"

# On enlève la virgule
df["Attendance"] = df["Attendance"].astype(str).str.replace(",", "")

# On convertit en nombre
df["Attendance"] = pd.to_numeric(df["Attendance"], errors="coerce")

# ==============================
# VÉRIFICATION DES TYPES APRÈS NETTOYAGE
# ==============================
df.info()
print("\n--------------------------------")
print("Nombre de lignes et de colonnes :", df.shape)

In [ ]:
# ==============================
# FEATURE ENGINEERING
# ==============================

# Trier les matchs par date
df = df.sort_values("Date")

# Réinitialiser l'index
df = df.reset_index(drop=True)

# Différence de buts
df["Goal_Diff"] = df["GF"] - df["GA"]

# Points obtenus selon le résultat
df["Points"] = df["Result"].map({
    "W": 3,
    "D": 1,
    "L": 0
})

# Indicateur domicile / extérieur
df["Is_Home"] = df["Venue"].map({
    "Home": 1,
    "Away": 0
})

# Numéro de journée
df["Matchweek"] = df["Round"].str.extract(r'(\d+)').astype(int)

# Points cumulés dans la saison
df["Points_Cum"] = df["Points"].cumsum()

# Différence de buts cumulée
df["Goal_Diff_Cum"] = df["Goal_Diff"].cumsum()

# Aperçu
df.head()

In [ ]:
# ==============================
# SAUVEGARDE DU DATASET PRÉPARÉ
# ==============================
# On enregistre la version enrichie du dataset dans le dossier processed

df.to_csv("../data/processed/annecy_matches_features.csv", index=False)